In [1]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install torch
!{sys.executable} -m pip install optuna
!{sys.executable} -m pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
%load_ext autotime

import os
import random
import gc
from itertools import combinations
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, log_loss
import joblib
import zipfile
import ast

time: 3.3 s (started: 2026-05-26 00:12:08 +09:00)


### File Setting

In [2]:
SEEDS = [42, 106, 1031]
RATIOS = [10]

def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

time: 2.8 ms (started: 2026-05-26 00:12:23 +09:00)


In [3]:
for seed in SEEDS: # SEEDS = [42, 106, 1031]
    print(f"\n========== SEED {seed} 실행 ==========")
    seed_everything(seed)

    train_df = pd.read_parquet('train.parquet')

    clicked_1 = train_df[train_df['clicked'] == 1]
    total_1 = len(clicked_1)

    for ratio in RATIOS:
        n_0 = total_1 * ratio
 
        clicked_0 = train_df[train_df['clicked'] == 0].sample(
            n=n_0, replace=True if n_0 > len(train_df[train_df['clicked']==0]) else False,
            random_state=seed
        )

        train_df_down = pd.concat([clicked_1, clicked_0], axis=0)\
                          .sample(frac=1, random_state=seed)\
                          .reset_index(drop=True)

        save_path = os.path.join('C:\\Users\\woolz\\Downloads\\open', f"train_xgb_{seed}_{ratio}.parquet")
        train_df_down.to_parquet(save_path, index=False)
        print(f"[SEED {seed} | Ratio {ratio}:1] Down-sampled Train saved at: {save_path} "
              f"(Class1: {len(clicked_1)}, Class0: {len(clicked_0)})")


========== SEED 42 실행 ==========
[SEED 42 | Ratio 10:1] Down-sampled Train saved at: C:\Users\woolz\Downloads\open\train_xgb_42_10.parquet (Class1: 204179, Class0: 2041790)

========== SEED 106 실행 ==========
[SEED 106 | Ratio 10:1] Down-sampled Train saved at: C:\Users\woolz\Downloads\open\train_xgb_106_10.parquet (Class1: 204179, Class0: 2041790)

========== SEED 1031 실행 ==========
[SEED 1031 | Ratio 10:1] Down-sampled Train saved at: C:\Users\woolz\Downloads\open\train_xgb_1031_10.parquet (Class1: 204179, Class0: 2041790)
time: 27min 10s (started: 2026-05-26 00:13:00 +09:00)


### File Load

In [6]:
base_path = "C:\\Users\\woolz\\Downloads\\open\\"
file_names = [
    "train_xgb_42_10.parquet",
    "train_xgb_106_10.parquet",
    "train_xgb_1031_10.parquet",
    "test.parquet"
]

datasets = {}
for i, fname in enumerate(file_names, start=1):
    df = pd.read_parquet(base_path + fname)
    if "test" in fname:
        key = "test"
    else:
        key = f"train_{i:02d}"
    datasets[key] = df
    print(f"[Loaded] {key} | shape={df.shape}")

# ========================
# 3. 접근 예시
# ========================
train_01 = datasets["train_01"]
train_02 = datasets["train_02"]
train_03 = datasets["train_03"]
test     = datasets["test"]

[Loaded] train_01 | shape=(2245969, 119)
[Loaded] train_02 | shape=(2245969, 119)
[Loaded] train_03 | shape=(2245969, 119)
[Loaded] test | shape=(1527298, 119)
time: 2min 25s (started: 2026-05-26 01:40:17 +09:00)


### Data Processing

In [7]:
def create_combination_features(df):

    base_cols = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']
    combo_features = {}

    for col_a, col_b in combinations(base_cols, 2): # combinations: base_cols 중 2개씩 뽑기
        combo_name = f'{col_a}_{col_b}'
        combo_features[combo_name] = (df[col_a].astype(str) + '_' + df[col_b].astype(str)).astype(object)

    combo_df = pd.DataFrame(combo_features, index=df.index) # index=df.index: 기존 df와 인덱스를 동일하게 맞추기
    df = pd.concat([df, combo_df], axis=1)

    print(f"✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})") # 총 10개의 feature 생성
    return df

train_01 = create_combination_features(train_01)
train_02 = create_combination_features(train_02)
train_03 = create_combination_features(train_03)
test     = create_combination_features(test)

✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
time: 16.2 s (started: 2026-05-26 01:42:51 +09:00)


In [12]:
df_click_prob = pd.read_excel('high_click_numbers.xlsx')

# zip으로 (428, 0.17) -> dict은 (key, value) 형태를 기대하므로 {428: 0.17}로 변환
click_prob_map = dict(zip(df_click_prob['number'], df_click_prob['click_prob']))

pos_list = {370, 528, 68, 561, 144, 227, 417, 442, 186, 395}
neg_list = {154, 222, 84, 498, 434, 511, 216, 497, 309, 446}

def add_seq_features(df, name="dataset"):
    seq_len, avg_prob, seq_neg, seq_pos = [], [], [], []

    for s in df["seq"]:
        if isinstance(s, str) and s != "": # 샘플 별 seq가 문자열인지, 빈 문자열은 아닌지 확인
            arr = [int(x) for x in s.split(",") if x]

            seq_len.append(len(arr))

            probs = [click_prob_map[num] for num in arr if num in click_prob_map] # 각 숫자에 대한 확률 lookup
            avg_prob.append(sum(probs) / len(probs) if probs else np.nan) # probs 없으면 결측값 처리

            seq_neg.append(sum(1 for x in arr if x in neg_list))
            seq_pos.append(sum(1 for x in arr if x in pos_list))
        else:
            seq_len.append(0)
            avg_prob.append(np.nan)
            seq_neg.append(0)
            seq_pos.append(0)

    df["seq_len"] = seq_len
    df["avg_click_prob"] = avg_prob
    df["seq_neglogcount"] = seq_neg
    df["seq_poslogcount"] = seq_pos

    print(f"✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_seq_features(train_01)
train_02 = add_seq_features(train_02)
train_03 = add_seq_features(train_03)
test     = add_seq_features(test)

✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
time: 26min 35s (started: 2026-05-26 01:51:18 +09:00)


In [13]:
full_train_df = pd.concat([train_01, train_02, train_03], ignore_index=True) # 기존 인덱스 버리고 다시 0부터 정렬

agg_targets = ['history_a_1','history_a_2','history_a_3', 'history_a_6','feat_d_4','l_feat_1','l_feat_2']

agg_stats = (
    full_train_df.groupby('inventory_id')[agg_targets]
    .agg(['mean','std'])
    .reset_index() # groupby 결과의 index(inventory_id)를 일반 column으로 변환
)

agg_stats.columns = ['inventory_id'] + [
    f'inventory_id_{col}_{stat}' for col, stat in agg_stats.columns[1:]
]

count_cols = ['age_group_inventory_id', 'inventory_id', 'inventory_id_hour']
count_maps = {col: full_train_df[col].value_counts().to_dict() for col in count_cols}

del full_train_df
gc.collect()


def add_features(df: pd.DataFrame, name: str = "dataset") -> pd.DataFrame:
    """groupby 통계량 + count encoding + 결측치 처리"""
    print(f"\n🚀 {name} 처리 시작...")

    df = df.merge(agg_stats, on='inventory_id', how='left')
    df.fillna(0, inplace=True)  # 숫자형 결측치 처리 (inplace=True: 원본 df 직접 수정)

    for col, cmap in count_maps.items():
        df[f"{col}_count"] = df[col].astype(str).map(cmap).fillna(0).astype(int)

    obj_cols = df.select_dtypes(include='object').columns
    df[obj_cols] = df[obj_cols].fillna("missing").astype(str)

    print(f"✅ {name}: seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_features(train_01)
train_02 = add_features(train_02)
train_03 = add_features(train_03)
test     = add_features(test)


🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)
time: 1min 22s (started: 2026-05-26 02:18:05 +09:00)


### Modeling

In [16]:
## 튜닝코드

# -----------------------
# 클래스 균형 가중치
# -----------------------
def make_class_balanced_weights(y_true):
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    w_pos = 0.5 / (n_pos if n_pos > 0 else 1)
    w_neg = 0.5 / (n_neg if n_neg > 0 else 1)
    return np.where(y_true == 1, w_pos, w_neg)

# -----------------------
# 전처리
# -----------------------
BASE_EXCLUDE = {"clicked", "ID", "seq"}

def preprocess(df, exclude_features=None):
    exclude = BASE_EXCLUDE.copy()
    if exclude_features:
        exclude |= set(exclude_features)
    X = df.drop(columns=[c for c in exclude if c in df.columns], errors="ignore")

    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category").cat.codes.astype("int32")
    for col in X.select_dtypes(include=["int64"]).columns:
        X[col] = X[col].astype("int32")
    for col in X.select_dtypes(include=["float64"]).columns:
        X[col] = X[col].astype("float32")
    return X

# -----------------------
# Optuna 튜닝 함수
# -----------------------
def tune_xgb_with_optuna(X, y, seed, n_trials=5):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    scale_pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))
    sampler = TPESampler(seed=seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def objective(trial):
        params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "device" : 'cuda',
            "scale_pos_weight": scale_pos_weight,
            "eta": trial.suggest_float("eta", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 200.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        }
        booster = xgb.train(
            params, dtrain,
            num_boost_round=1000,
            evals=[(dval, "valid")],
            early_stopping_rounds=30,
            verbose_eval=False
        )
        p_valid = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
        ap = average_precision_score(y_val, p_valid)
        wll = log_loss(y_val, p_valid, sample_weight=make_class_balanced_weights(y_val), labels=[0, 1])
        final_score = 0.5 * ap + 0.5 * (1.0 / (1.0 + wll))
        return final_score

    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "device" : "cuda",
        "scale_pos_weight": float((y == 0).sum() / max(1, (y == 1).sum()))
    })
    return best_params

# -----------------------
# 학습+예측 함수
# -----------------------
def train_and_predict(train_df, test_df, id_tag, seed, n_trials=50):
    print(f"\n=== START PIPELINE: {id_tag} | seed={seed} ===")

    random.seed(seed)
    np.random.seed(seed)

    y = train_df["clicked"].astype(int).values.ravel()
    X = preprocess(train_df)
    X_test = preprocess(test_df)
    test_ids = test_df["ID"].values if "ID" in test_df.columns else np.arange(len(test_df))

    # Optuna 탐색
    print(f"[{id_tag}] Optuna tuning ...")
    best_params = tune_xgb_with_optuna(X, y, seed=seed, n_trials=n_trials)
    print(f"[{id_tag}] Best params:", best_params)

    # 최종 학습
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    print(f"[{id_tag}] Training final model ...")
    model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=50,
        verbose_eval=100
    )

    # 모델 저장
    model_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', f"model_{id_tag}.pkl")
    joblib.dump(model, model_path)
    print(f"[{id_tag}] Model saved -> {model_path} | best_iter={model.best_iteration}")

    # 테스트 예측
    dtest = xgb.DMatrix(X_test)
    preds = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
    sub = pd.DataFrame({"ID": test_ids, "clicked": preds})
    csv_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', f"submission_{id_tag}.csv")
    sub.to_csv(csv_path, index=False)
    print(f"[{id_tag}] Submission saved -> {csv_path} | shape={sub.shape}")
    print(f"=== END PIPELINE: {id_tag} ===\n")
    return model_path, csv_path

# -----------------------
# 실행
# -----------------------
datasets = [
    {"df": train_01, "id": "train_01", "seed": 106},
    {"df": train_02, "id": "train_02", "seed": 1031},
    {"df": train_03, "id": "train_03", "seed": 42},
]

results = {}
for data in datasets:
    model_path, csv_path = train_and_predict(
        train_df=data["df"],
        test_df=test,
        id_tag=data["id"],
        seed=data["seed"],
        n_trials=50
    )
    results[data["id"]] = {"model": model_path, "submission": csv_path}

print("=== ALL DONE ===")
print(results)


=== START PIPELINE: train_01 | seed=106 ===
[train_01] Optuna tuning ...


[I 2026-05-26 03:06:40,170] A new study created in memory with name: no-name-b3ab1959-28e6-4e35-9208-c02fb000cd5d
[I 2026-05-26 03:08:16,829] Trial 0 finished with value: 0.45652911684461783 and parameters: {'eta': 0.010259898367843918, 'max_depth': 12, 'subsample': 0.7798260137416737, 'colsample_bytree': 0.8440875086908464, 'min_child_weight': 170.55990367758548, 'gamma': 3.6940510569846796, 'lambda': 5.886868084847547, 'alpha': 0.5213830060758549}. Best is trial 0 with value: 0.45652911684461783.
[I 2026-05-26 03:09:09,477] Trial 1 finished with value: 0.4299307811965493 and parameters: {'eta': 0.12969245549678007, 'max_depth': 11, 'subsample': 0.7409115664353482, 'colsample_bytree': 0.6848127138204806, 'min_child_weight': 156.30843112731282, 'gamma': 4.4970562619261045, 'lambda': 2.6894091077470376e-08, 'alpha': 1.725942108719862e-07}. Best is trial 0 with value: 0.45652911684461783.
[I 2026-05-26 03:09:29,931] Trial 2 finished with value: 0.4471420995700206 and parameters: {'eta': 

[train_01] Best params: {'eta': 0.01618376768536903, 'max_depth': 11, 'subsample': 0.8180087139438176, 'colsample_bytree': 0.7461805665212552, 'min_child_weight': 199.77586286687585, 'gamma': 1.5688536673970268, 'lambda': 0.00015362425529485643, 'alpha': 1.9522758167149342e-05, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_01] Training final model ...
[0]	train-logloss:0.69052	valid-logloss:0.69058
[100]	train-logloss:0.59028	valid-logloss:0.59492
[200]	train-logloss:0.56954	valid-logloss:0.57790
[300]	train-logloss:0.55972	valid-logloss:0.57058
[400]	train-logloss:0.55304	valid-logloss:0.56581
[500]	train-logloss:0.54739	valid-logloss:0.56197
[600]	train-logloss:0.54253	valid-logloss:0.55872
[700]	train-logloss:0.53811	valid-logloss:0.55580
[800]	train-logloss:0.53374	valid-logloss:0.55291
[900]	train-logloss:0.52952	valid-logloss:0.55013
[999]	train-logloss:0.52565	valid-logloss:0.54763
[train_01] M

[I 2026-05-26 03:54:16,340] A new study created in memory with name: no-name-10bc0ea0-8505-408f-ba18-d3c3237d480f
[I 2026-05-26 03:54:46,227] Trial 0 finished with value: 0.4498155768228451 and parameters: {'eta': 0.010030542235435747, 'max_depth': 6, 'subsample': 0.7299275357796907, 'colsample_bytree': 0.7672919056877435, 'min_child_weight': 154.42315013332836, 'gamma': 4.788695606033571, 'lambda': 5.838643100318619e-07, 'alpha': 0.0010984897811838177}. Best is trial 0 with value: 0.4498155768228451.
[I 2026-05-26 03:55:38,475] Trial 1 finished with value: 0.41889721029711524 and parameters: {'eta': 0.11978434297771987, 'max_depth': 10, 'subsample': 0.8028360964022752, 'colsample_bytree': 0.9033177780841783, 'min_child_weight': 17.555385408203744, 'gamma': 3.2365936437527756, 'lambda': 3.0923631972991695e-05, 'alpha': 0.1762792686327148}. Best is trial 0 with value: 0.4498155768228451.
[I 2026-05-26 03:56:00,659] Trial 2 finished with value: 0.454403640509227 and parameters: {'eta': 0

[train_02] Best params: {'eta': 0.01194009328597376, 'max_depth': 12, 'subsample': 0.8993406937311019, 'colsample_bytree': 0.7938513318230591, 'min_child_weight': 187.91416931034715, 'gamma': 4.382320091632367, 'lambda': 1.4412789662646885e-05, 'alpha': 0.00787976663026755, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_02] Training final model ...
[0]	train-logloss:0.69109	valid-logloss:0.69115
[100]	train-logloss:0.59562	valid-logloss:0.60126
[200]	train-logloss:0.56895	valid-logloss:0.57898
[300]	train-logloss:0.55496	valid-logloss:0.56851
[400]	train-logloss:0.54678	valid-logloss:0.56267
[500]	train-logloss:0.54057	valid-logloss:0.55835
[600]	train-logloss:0.53521	valid-logloss:0.55472
[700]	train-logloss:0.53075	valid-logloss:0.55176
[800]	train-logloss:0.52646	valid-logloss:0.54892
[900]	train-logloss:0.52225	valid-logloss:0.54614
[999]	train-logloss:0.51846	valid-logloss:0.54363
[train_02] Model

[I 2026-05-26 04:50:32,736] A new study created in memory with name: no-name-312fbbca-b668-484f-9462-8c0bc7017f8d
[I 2026-05-26 04:51:35,626] Trial 0 finished with value: 0.4473676226646147 and parameters: {'eta': 0.030710573677773714, 'max_depth': 12, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'min_child_weight': 32.04770944804487, 'gamma': 0.7799726016810132, 'lambda': 3.3323645788192616e-08, 'alpha': 0.6245760287469893}. Best is trial 0 with value: 0.4473676226646147.
[I 2026-05-26 04:52:10,745] Trial 1 finished with value: 0.4490541045503113 and parameters: {'eta': 0.06054365855469246, 'max_depth': 10, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 166.65608551928392, 'gamma': 1.0616955533913808, 'lambda': 4.329370014459266e-07, 'alpha': 4.4734294104626844e-07}. Best is trial 1 with value: 0.4490541045503113.
[I 2026-05-26 04:52:37,706] Trial 2 finished with value: 0.4549618275161664 and parameters: {'eta': 0

[train_03] Best params: {'eta': 0.016879565866360594, 'max_depth': 11, 'subsample': 0.9382128272541636, 'colsample_bytree': 0.8186058748926905, 'min_child_weight': 176.15231003281565, 'gamma': 1.7455198366465, 'lambda': 1.143217533091322e-08, 'alpha': 0.04849712814060122, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_03] Training final model ...
[0]	train-logloss:0.69036	valid-logloss:0.69043
[100]	train-logloss:0.58810	valid-logloss:0.59393
[200]	train-logloss:0.56601	valid-logloss:0.57641
[300]	train-logloss:0.55464	valid-logloss:0.56809
[400]	train-logloss:0.54751	valid-logloss:0.56314
[500]	train-logloss:0.54189	valid-logloss:0.55933
[600]	train-logloss:0.53641	valid-logloss:0.55568
[700]	train-logloss:0.53152	valid-logloss:0.55244
[800]	train-logloss:0.52708	valid-logloss:0.54956
[900]	train-logloss:0.52233	valid-logloss:0.54646
[999]	train-logloss:0.51811	valid-logloss:0.54374
[train_03] Model s

In [17]:
# ✅ 1) 파일 불러오기 (Drive에서 불러오기)
df1 = pd.read_csv(os.path.join("submission_train_01.csv"))
df2 = pd.read_csv(os.path.join("submission_train_02.csv"))
df3 = pd.read_csv(os.path.join("submission_train_03.csv"))

# ✅ 2) ID 기준 병합
merged = df1.merge(df2, on="ID", suffixes=("_1", "_2"))
merged = merged.merge(df3, on="ID")
merged.rename(columns={"clicked": "clicked_3"}, inplace=True)

# ✅ 3) Soft Voting (평균)
merged["clicked"] = (
    merged["clicked_1"] + merged["clicked_2"] + merged["clicked_3"]
) / 3

# ✅ 4) ID와 최종 clicked만 추출
final = merged[["ID", "clicked"]]

# ✅ 5) 저장 (→ /content 폴더에 저장)
output_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', "xgb.csv")
final.to_csv(output_path, index=False)

print(f"🎯 Soft Voting 결과 저장 완료!\n파일 경로: {output_path}")
print(final.head())

🎯 Soft Voting 결과 저장 완료!
파일 경로: C:\Users\woolz\Downloads\open\xgb.csv
             ID   clicked
0  TEST_0000000  0.312046
1  TEST_0000001  0.272580
2  TEST_0000002  0.351088
3  TEST_0000003  0.472659
4  TEST_0000004  0.289522
time: 5.38 s (started: 2026-05-26 06:54:09 +09:00)
